# Analysis notebook for PiC_UVnudge runs
## Set up
### Packages

In [1]:
import numpy as np
import xarray as xr
import xesmf as xe
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import pandas as pd
import scipy
from scipy import stats, interpolate
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.mathtext import _mathtext as mathtext
import matplotlib.ticker as mticker
from matplotlib import gridspec, animation
import matplotlib.path as mpath
import matplotlib.colors as colors
import matplotlib.dates as mdates
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from cmcrameri import cm
from Processing_functions import FixLongitude, Wilks_pcrit, lats, arclats, lons
import jinja2
import cftime
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from functools import partial
from collections import defaultdict
import os
import copy

In [2]:
## Plot types to make - CHANGE
# 0: Weighted spatial mean or Arctic sea ice area
# 1: Spatial maximum (AMOC)
# 2: Leave alone (doing spatial map or sea ice concentration)
# 3: Zonal
plots = {
    'amoc': [False, 1],
    'lin': [False, 0],
    'map': [False, 2],
    'mtrd': [True, 0],
    'strd': [False, 2],
    'zon': [False, 3],
    'ztrd': [False, 3]
}

## Categorical plot type - DO NOT CHANGE
plot_types = {
    'spatial': [False, []],
    'line': [False, []],
    'mtrd': [False, []],
    'zonal': [False, []]
}

# Set up plot_types based on plots
for pl, att in plots.items():
    if att[0]:
        if pl == 'mtrd':
            plot_types['mtrd'][0] = True
            plot_types['mtrd'][1].append(pl)
        elif att[1] <= 1:
            plot_types['line'][0] = True
            plot_types['line'][1].append(pl)
        elif att[1] == 2:
            plot_types['spatial'][0] = True
            plot_types['spatial'][1].append(pl)
        elif att[1] == 3:
            plot_types['zonal'][0] = True
            plot_types['zonal'][1].append(pl)

# Spatial domain - CHANGE s_domain & t_domain only
s_domain = True # True: Global, False: Arctic
r_domain = False # True: use Arctic regional domain, False: pan-Arctic average
a_domain = plot_types['spatial'][0] or plot_types['zonal'][0] # True: 50-90, False: 70-90
o_domain =  True # True: 20-60N max at each lat, False: 33-55N max
subset_time = False # True: slice time in MasterDS, False: Need all data post-1950 for XX-yr trends
t_domain = 1950 # start year
trd_length = 20 # trend length

## Time averaging type - CHANGE
time_avg = 4  # 0: Monthly, 1: Yearly, 2: Seasonal, 3: All data, 4: Timeseries

## Ensemble mean or All members - CHANGE
ens_type = 0   # 0: All_members, 1: Mean

# Variables - CHANGE
comp = 'ocn' # compset
freq = 0 # 0: monthly, 1: daily
var_ind = 0

# Plot levels for spatial trends - CHANGE
plot_levels = [300, 500, 850, 925]

In [3]:
# DO NOT CHANGE
var_list = {'atm': ['TREFHT','PSL','RESTOM','Z3','FSNTOA','TGCLDLWP','AEROD_v'],
            'ice': ['aice','hi'],
            'ocn': ['MOC','TEMP']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]

In [4]:
## Test numbers - DO NOT CHANGE
tst_nums = np.arange(1,4)

## Test names
# O: LENS ensemble
# 1: PiC_UVnudge ensemble
# 2: HIST_UVnudge ensemble
# 3: observations
# attribute structure: [use dataset, dataset type, line color, line style, zorder]
# CHANGE ONLY - use dataset
use_era5 = var in ['aice','TREFHT','Z3']
use_ceres = var in ['RESTOM','FSNTOA','TOT_CLD_VISTAU']
ds_names = {
    'LENS2 piControl': [True, 0],
    'LENS2 HIST-SSP370': [True, 0],
    'CESM2-lessmelt HIST': [False, 0],
    'GHG2': [True, 0],
    'PiC_UVnudge_LM2006': [True, 1],
    'HIST_UVnudge_LM': [True, 2],
    'GHG_UVnudge_LM': [True, 1],
    'ERA5': [use_era5, 3],
    'HadCRUT5': [var == 'TREFHT' and plots['mtrd'][0], 3],
    'BEST': [var == 'TREFHT' and plots['mtrd'][0], 3],
    'GISTEMP': [var == 'TREFHT' and (plots['mtrd'][0] or plots['lin'][0]), 3],
    'NSIDC': [var == 'aice' and (plots['mtrd'][0] or plots['lin'][0]), 3],
    'OSISAF': [var == 'aice' and (plots['mtrd'][0] or plots['lin'][0]), 3],
    'DA_SIC_RECON': [var == 'aice' and (plots['mtrd'][0] or plots['lin'][0]), 3],
    'RAPID': [var == 'MOC' and (plots['mtrd'][0] or plots['amoc'][0]), 3],
    'CERES': [use_ceres, 3]
}
vercompres = 'b.e21.B1850cmip6.f09_g17.'
cesm2le = 'b.e21.BHISTsmbb.f09_g17.'
cesm2piC = 'b.e21.B1850.f09_g17.'
le_names = {'LENS2 piControl': cesm2piC+'CMIP6-piControl.001',
            'CESM2-lessmelt HIST': cesm2le+'LM-smbb.*',
            'LENS2 HIST-SSP370': cesm2le+'LE-smbb.*',
            'GHG2': vercompres+'SF2-GHG.*'}

## Paper names - DO NOT CHANGE
ds_paper_names = {
    'CESM2-lessmelt HIST': 'HIST-lessmelt-control',
    'LENS2 HIST-SSP370': 'HIST-SSP370-control',
    'GHG2': 'GHG-control',
    'LENS2 piControl': 'PI-control',
    'PiC_UVnudge_LM2006': 'PI+winds',
    'ERA5': 'OBS (ERA5)',
    'GISTEMP': 'OBS (GISTEMP)',
    'HadCRUT5': 'OBS (HadCRUT5)',
    'BEST': 'OBS (BEST)',
    'NSIDC': 'OBS (NSIDC)',
    'OSISAF': 'OBS (OSISAF)',
    'DA_SIC_RECON': 'OBS (DA SIC)',
    'RAPID': 'OBS (RAPID)',
    'CERES': 'OBS (CERES)',
    'HIST_UVnudge_LM': 'HIST+winds',
    'GHG_UVnudge_LM': 'GHG+winds',
}


## Filepaths - DO NOT CHANGE
path_to_work = '/glade/work/glydia/'
path_to_lensdata = path_to_work+'processed_CESM2_LENS_data/longer_recent/'
path_to_expdata = path_to_work+'Arctic_controls_processed_data/'
path_to_plotdata  = path_to_expdata+'recent_plotting_data/'

# Extensions - DO NOT CHANGE
h_ext = {'atm': ['.h0.'],
       'ice': ['.h.','.h1.'],
       'ocn': ['.h.']}
yr_extn = {False: [".195001-202412.",".19500101-20241231."],
           True: [".*.",".05010101-05741231."]}
vert_lev = {'atm': [False,False,False,True,False,False,False],
            'ice': [False,False],
            'ocn': [False,False]}
file_bool = not vert_lev[comp][var_ind] and freq == 0
file_ext = {True: 'nc', False: 'zarr'}

In [5]:
########################## DO NOT CHANGE ANYTHING BELOW THIS LINE #############################

In [6]:
%%time
    
## Select plot type
time_str_list = {0: 'month', 1: 'year', 2: 'season', 3: 'all', 4: 'timeseries'}
time_outstr = time_str_list[time_avg]

## Select ensemble type
ens_str_list = {0: 'All_members', 1: 'Mean'}
ens_str = ens_str_list[ens_type]

## Select time and spatial domain strings
s_domain = True if ((var == 'RESTOM' and plots['lin'][0]) or comp == 'ocn') else s_domain # Make sure TOA is global domain
sd_str_list = {True: 'Global', False: 'Arctic'}
sd_str = sd_str_list[s_domain]

rd_str_list = {True: 'Reg', False: ''}
rd_str = rd_str_list[r_domain]

od_str_list = {True: 'Lat', False: ''}
od_str = od_str_list[o_domain]

td_str = str(t_domain)
t_end = 2024

CPU times: user 7 µs, sys: 1 µs, total: 8 µs
Wall time: 11.7 µs


In [7]:
## Print Script Configurations
# Plot types
print('Plot types: ')
catcount = 0

for cat, att in plot_types.items():
    if att[0]:
        print('   '+cat)
        catcount += 1
        for pl in att[1]:
            print('      '+pl)

# Variable
if var == 'Z3':
    print('Variable: Z3, U, V')
else:
    print('Variable(s): '+var)

# Datasets
print('Datasets:')
for dsname, attr in ds_names.items():
    if attr[0]:
        print('   '+dsname)

# Spatial domain
print('Spatial domain: '+sd_str)

# Regional division
print('Regional domains: '+('Yes' if r_domain else 'No'))

# Ocean latitude
print('Ocean domain: '+('20-60N' if o_domain else '33-55N'))

# Subset time
print('Subsetting time: '+('Yes' if subset_time else 'No'))
if not subset_time:
    print('Trend length: '+str(trd_length)+' years')

# Time domain
print('Time domain: '+td_str+'-'+str(t_end))

# Time averaging
print('Time averaging: '+time_outstr)

# Ensemble
print('Ensemble averaging: '+ens_str)

Plot types: 
   mtrd
      mtrd
Variable(s): MOC
Datasets:
   LENS2 piControl
   LENS2 HIST-SSP370
   GHG2
   PiC_UVnudge_LM2006
   HIST_UVnudge_LM
   GHG_UVnudge_LM
   RAPID
Spatial domain: Global
Regional domains: No
Ocean domain: 20-60N
Subsetting time: No
Trend length: 20 years
Time domain: 1950-2024
Time averaging: timeseries
Ensemble averaging: All_members


In [8]:
## Check number of different averaging types
if catcount > 1:
    raise SystemExit("ERROR: More than one averaging type fed into CreateMasterDS()")

### Cluster

In [9]:
cluster = PBSCluster(cores    = 1,
                     memory   = '35GiB',
                     queue    = 'casper',
                     walltime = '12:00:00',
                     account  = 'UCUB0137',
                     log_directory = '/glade/u/home/glydia/python/logs/',
                     name='PiC_HIST_UVnudge_process_'+var)
cluster.scale(4*9)
client = Client(cluster)

In [10]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.116:36131,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/glydia/Arctic_breakdown/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


### Custom functions

#### pre_process

In [11]:
def pre_process(da):
    da = da.assign_coords(time=pd.date_range('1950-01-01',str(t_end)+'-12-01',freq='MS'))
    return da

#### pre_processlm

In [12]:
def pre_processlm(da):
    da = da.assign_coords(time=pd.date_range('1990-01-01','2014-12-01',freq='MS'))
    return da

#### LoadData

In [13]:
def LoadData(casename, set_type, varname):
    # Condition for only using 501-574 from LENS2 piControl, i.e. when plotting Z3
    cond_lens = var != 'Z3' # True means use 50 slices, False means use 501-574 only
    
    # Create file name
    if set_type == 0:
        # If cond_lens true, select all netcdf files with *
        # Else, select zarr file with 1950-2024 time string
        filename = le_names[casename]+h_ext[comp][freq]+varname+yr_extn[cond_lens][freq]+file_ext[file_bool]
        totalpath = path_to_lensdata+filename
    elif set_type == 3:
        filename = casename+h_ext[comp][freq]+varname+yr_extn[False][freq]+file_ext[file_bool]
        totalpath = path_to_work+'processed_'+casename+'_data/'+filename
    elif set_type == 2:
        filename = cesm2le+casename+h_ext[comp][freq]+varname+yr_extn[False][freq]+file_ext[file_bool]
        totalpath = path_to_expdata+'processed_'+casename+'_data/'+filename
    else:
        filename = vercompres+casename+h_ext[comp][freq]+varname+yr_extn[False][freq]+file_ext[file_bool]
        totalpath = path_to_expdata+'processed_'+casename+'_data/'+filename

    # Load if piControl and using slices
    if set_type == 0 and cond_lens: 
        if casename == 'CESM2-lessmelt HIST':
            data = xr.open_mfdataset(totalpath, combine='nested',concat_dim='slice',preprocess=pre_processlm)
        else:
            data = xr.open_mfdataset(totalpath, combine='nested',concat_dim='slice',preprocess=pre_process)  
    
    # Load if netCDF and not piControl
    elif file_bool:
        data = xr.open_dataset(totalpath, chunks={'time':12})  

    # Load if Zarr
    else:
        data = xr.open_zarr(totalpath, group=varname,  chunks={'time':12})

    return data

In [14]:
# Set spatial domain slicing
slice_ocn = dict(lat_aux_grid=slice(33,55),moc_z=slice(800*100,2200*100), moc_comp=0,transport_reg=1)
slice_ocnN = dict(lat_aux_grid=slice(20,60),moc_z=slice(800*100,2200*100), moc_comp=0,transport_reg=1)
slice_icemod = dict(nj=slice(250,385))
slice_iceobs = dict(lat=slice(50,90))
slice_atmwei = dict(lat=slice(70,90))
slice_atmspt = dict(lat=slice(50,90))
slice_time = dict(time=slice(td_str+'-01-01',str(t_end)+'-12-31'))
slice_rdom = {'HA': dict(lat=slice(80,90)),
              'G': dict(lat=slice(66.5,90),lon=slice(-70,0)),
              'AA': dict(lat=slice(66.5,90),lon=slice(0,60)),
              'WS': dict(lat=slice(66.5,90),lon=slice(60,120)),
              'ES': dict(lat=slice(66.5,90),lon=slice(120,180)),
              'PA': dict(lat=slice(66.5,90),lon=slice(-180,-70))}

#### CreateMasterDS

In [15]:
def CreateMasterDS(varname):
    ds_load_list = []
    pd_time = pd.date_range('1950-01-01',str(t_end)+'-12-01',freq='MS')
    pd_stime = pd.date_range('1950-01-01','2023-12-01',freq='MS')
    pd_lmtime = pd.date_range('1990-01-01','2014-12-01',freq='MS')

    # Find which plots will be made
    funcstr = None
    for plotname, attrs in plots.items():
        # If plot will be made
        if attrs[0]:
            # If weighted spatial average or Arctic sea ice area
            if attrs[1] == 0:
                if varname == 'aice':
                    funcstr = 'S'
                elif varname == 'MOC':
                    funcstr = 'M'
                else:
                    funcstr = 'W'
                break
            # Elif spatial maximum
            elif attrs[1] == 1:
                funcstr = 'M'
                break
            # Elif zonal average
            elif attrs[1] == 3:
                funcstr = 'Z'
                break

    print('Figure out processing calculations to do')

    ## Load all datasets
    # Load each dataset
    for dsname, attrs in ds_names.items():
        # If dataset is going to be plotted
        if attrs[0]:
            print(dsname)
            # If dataset is ERA5
            if dsname == 'ERA5':
                # If using U or V variable, use Target_U & Target_V from PiC_UVnudge
                if varname == 'U' or varname == 'V':
                    varname = 'Target_'+varname
                    ds_load = LoadData('PiC_UVnudge', 1, varname)
                    ds_load = ds_load.mean('ensemble_member')
                else:
                    ds_load = LoadData(dsname, attrs[1], varname)
                    ds_load = CalcGridArea(ds_load)
            else:
                ds_load = LoadData(dsname, attrs[1], varname)
                if dsname == 'DA_SIC_RECON':
                    ds_load = ds_load.rename({'gridarea_sqkm':'area'})
    
            # Reduce to dataarray
            ds_load = ds_load[varname]
        
            # Rename dataarray name to be casename
            ds_load = ds_load.rename(dsname)
            
            print('  Initial data loading complete')

            # Make all datasets have the same time axis
            if dsname == 'CESM2-lessmelt HIST':
                ds_load = ds_load.assign_coords(time=pd_lmtime)
            elif dsname == 'DA_SIC_RECON':
                ds_load = ds_load.assign_coords(time=pd_stime)
            elif dsname != 'NSIDC' and dsname != 'CERES' and dsname != 'OSISAF' and dsname != 'RAPID':
                ds_load = ds_load.assign_coords(time=pd_time)
            else:
                stime = ds_load.time[0]
                etime = ds_load.time[-1]
                pd_ctime = pd.date_range(str(stime.dt.year.values)+'-'+str(stime.dt.month.values).zfill(2)+'-01',
                                        str(etime.dt.year.values)+'-'+str(etime.dt.month.values).zfill(2)+'-01',
                                        freq='MS')
                ds_load = ds_load.assign_coords(time=pd_ctime)

            print('  Fixed time dimension')
                

            ## Do weighted averages, ensemble means, sea ice area calculations, and area maximums
            # Do spatial domain slicing
            # ICE
            if comp == 'ice' and dsname != 'NSIDC' and dsname != 'OSISAF':
                ds_load = ds_load.loc[slice_iceobs] if attrs[1] == 3 else ds_load.loc[slice_icemod]
            # OCEAN
            elif comp == 'ocn' and attrs[1] != 3:
                ds_load = ds_load.loc[slice_ocnN].drop('moc_components') if o_domain else ds_load.loc[slice_ocn].drop('moc_components')
            # ATMOSPHERE
            elif comp == 'atm' and (not s_domain and not r_domain):
                # Make sure all atm variables have same lat/lon values
                ds_load = ds_load.assign_coords({'lat':lats,'lon':lons}) if attrs[1] != 3 else ds_load
                ds_load = ds_load.loc[slice_atmspt] if a_domain else ds_load.loc[slice_atmwei]
            
                
            print('  Sliced data')
                    
            # Sort processing by variable and graph type
            # Doing spatial plot or SIC
            if funcstr == None or funcstr == 'Z':
                print('  No proceesing, spatial')
                # If ERA5, rename lat/lon
                if dsname == 'ERA5':
                    ds_load = ds_load.rename({'lat':'latE', 'lon':'lonE'})
                    if varname == 'Z3':
                        ds_load = ds_load.rename({'lev':'plev'})

                if funcstr == 'Z':
                    print('  Calculating zonal average')
                    if dsname == 'ERA5':
                        ds_load = ds_load.mean('lonE', skipna=True)
                    else:
                        ds_load = ds_load.mean('lon', skipna=True)
                else:
                    print('  No proceesing, spatial')
                
                    
            # Doing SIA plot(s)
            elif funcstr == 'S':
                print('  Calculating sea ice extent')
                if dsname != 'NSIDC' and dsname != 'OSISAF':
                    ds_load = CalcSIE(ds_load, attrs[1])
            # Doing ts or TOA plot(s)
            elif funcstr == 'W':
                print('  Calculating weighted average')
                ds_load = CalcRegionalAvg(ds_load) if r_domain else CalcWeightedMean(ds_load)
            # Doing AMOC plots:
            elif funcstr == 'M' and attrs[1] != 3:
                print('  Calculating maximum')
                if o_domain:
                    ds_load = ds_load.max('moc_z')
                    ds_load = ds_load.rename({'lat_aux_grid':'lat'})
                else:
                    ds_load.max(('moc_z','lat_aux_grid'))
                
            # If Ensemble mean
            if ens_type and (attrs[1] == 1 or attrs[1] == 2):
                print('  Calculating ensemble mean')
                ds_load = ds_load.mean('ensemble_member')

            ds_load = ds_load.rename(dsname)

            print('  Processing on dataset complete')
            ds_load_list.append(ds_load)

            # Add LENS2 ensemble mean if not 3D variable
            if attrs[1] == 0:
                ds_load_mean = ds_load.mean('slice')
                ds_load_mean = ds_load_mean.rename(dsname+' mean')
                ds_load_list.append(ds_load_mean)
    
    # Merge dataarrays
    ds_proc = xr.merge(ds_load_list)

    print('All datasets merged')

    # Slice time
    ds_proc = ds_proc.loc[slice_time]
    
    return ds_proc

#### CalcWeightedMean

In [16]:
def CalcWeightedMean(ds):
    # Set up
    avg_dim = ('lon','lat')

    # Create weights
    weights = np.cos(np.deg2rad(ds.lat))
    weights.compute()

    # Weight data
    ds_w = ds.weighted(weights)

    # Calculate weighted mean
    ds_mean_w = ds_w.mean(avg_dim, skipna=True)

    ds_mean_w.compute()
    
    return ds_mean_w

#### CalcRegionalAvg

In [17]:
def CalcRegionalAvg(ds):
    # Set up lists
    ds_regavg_list = []
    dom_regname_list = []

    # Cycle over all regions
    for dname, domain in slice_rdom.items():
        da_dom = ds.loc[domain]

        # Calculate weighted average for region
        da_domw = CalcWeightedMean(da_dom)

        # Add to lists
        ds_regavg_list.append(da_domw)
        dom_regname_list.append(dname)

    ds_dom = xr.concat(ds_regavg_list, dim=pd.Index(dom_regname_list, name='region'))
    return ds_dom

#### CalcAnnSeasMean

In [18]:
def CalcAnnSeasMean(ds, atype):
    # atype = 1: annual mean, atype = 2: seasonal mean
    if atype == 1:
        cdim = 'year'
    elif atype == 2:
        cdim = 'season'
    adim = 'time.'+cdim 
    
    # Create weights
    month_length = ds.time.dt.days_in_month
    month_weights = month_length.groupby(adim)/month_length.groupby(adim).sum()

    # Calculate mean for each year
    ds_grp = ds.groupby('time.year')
    month_weights_grp = month_weights.groupby('time.year')

    grp_avg_list = []
        
    for yr, dgrp in ds_grp:
        if atype == 1:
            # Weight data
            dgrp = dgrp.weighted(month_weights_grp[yr])
            
            # Calculate annual mean
            davg = dgrp.mean('time', skipna=True)

        elif atype == 2:
            mw_dgrp = month_weights_grp[yr]
            davg = (dgrp * mw_dgrp).groupby("time.season").sum(dim="time")
            
        davg = davg.expand_dims({'year':[yr]})
        grp_avg_list.append(davg)

    ds_avg = xr.concat(grp_avg_list,dim='year')
    return ds_avg

#### CalcAnom

In [19]:
def CalcAnom(ds_mean_w, period):
    # Calculate period mean
    ds_p_mean_w = ds_mean_w.mean(period,skipna=True)

    # Calculate anomalies
    ds_anom_w = ds_mean_w-ds_p_mean_w

    return ds_anom_w       

#### CalcDetAnom

In [20]:
def CalcDetAnom(ds_anom_w, period):
    if period == 'time':
        raw_time = ds_anom_w.time  
        ds_anom_w = AddCoordTrend(ds_anom_w, period)
        
    # Calculate linear regression coefficients
    ds_reg_coef = ds_anom_w.polyfit(dim=period,deg=1)

    # Calculate y-values of linear regression
    ds_yreg = (ds_reg_coef.loc[dict(degree=1)]*ds_anom_w[period])+ds_reg_coef.loc[dict(degree=0)]

    # Calculate detrended anomalies
    ds_dtrd_anom = ds_anom_w-ds_yreg['polyfit_coefficients']

    if period == 'time':
        ds_dtrd_anom = ds_dtrd_anom.assign_coords(time=raw_time)
    
    return ds_dtrd_anom

#### CalcR

In [21]:
def CalcR(ds_obs, ds_mod, period, ens):
    if ens == 1 and not ens_type:
        # Calculate model ensemble mean
        ds_mod_mean = ds_mod.mean('ensemble_member')
    else:
        ds_mod_mean = ds_mod

    if ens == 4:
        slice_short = slice_year_short if time_avg == 1 else slice_time_short
        ds_obs_mod = ds_obs.loc[slice_short]
    else:
        ds_obs_mod = ds_obs
        
    # Calculate correlation coefficient for anomalies (detrended and not)
    ds_r = xr.corr(ds_obs_mod, ds_mod_mean, dim=period)
    
    return ds_r

#### CalcEnsSp

In [22]:
def CalcEnsSp(ds_mean_w):
    # Calculate ensemble spread
    ds_ensp = np.sqrt(ds_mean_w.var('ensemble_member'))

    return ds_ensp

#### CalcLEMinMax

In [23]:
def CalcLEMinMax(ds):
    for dname, attrs in ds_names.items():
        if attrs[0] and attrs[1] == 0:
            ds[dname+' min'] = ds[dname].min('slice')
            ds[dname+' max'] = ds[dname].max('slice')

#### DropNonPiC

In [24]:
def DropNonPiC(ds):
    # Drop all variables and dimensions not used by PiC_UVnudge
    var_set = set(ds.keys())

    non_pic_vars = []
    non_pic_dims = []
    for v in var_set:
        if 'PiC_UVnudge' not in v:
            non_pic_vars.append(v)

            if v == 'LENS2 piControl':
                non_pic_dims.append('slice')
        
    ds_d = ds.drop_vars(non_pic_vars)
    ds_d = ds_d.drop_dims(non_pic_dims)

    return ds_d

#### CalcMonthTrd

In [25]:
def CalcMonthTrd(weighted_avg):
    # Group by month & calculate trends
    grouped = weighted_avg.groupby('time.month')

    ds_list = []
    dsp_list = []
    for mon, dsmon in grouped:
        ds_sizes = dsmon.sizes
        new_time = np.arange(1,ds_sizes['time']+1)
        
        dsmon = dsmon.assign_coords(time=new_time)
        dsmon = dsmon.polyfit(dim='time',deg=1)
        dsmon *= 10
        ds_list.append(dsmon)

    month_trd = xr.concat(ds_list, pd.Index(np.arange(1,13),name='month'))
    
    return month_trd['polyfit_coefficients'].loc[dict(degree=1)]

#### CalcMonthTrd2ndOrder

In [26]:
# def CalcMonthTrd2ndOrder(ds, ds_fstord):
#     ds['Anthro'] = ds['HIST_UVnudge_LM']-ds['PiC_UVnudge_LM2006']
#     ds['GHG'] = ds['GHG_UVnudge_LM']-ds['PiC_UVnudge_LM2006']
#     ds['Non-GHG'] = ds['HIST_UVnudge_LM']-ds['GHG_UVnudge_LM']
#     ds_fstord['Anthro'] = ds_fstord['HIST_UVnudge_LM']-ds_fstord['PiC_UVnudge_LM2006']
    
#     # Group by month & calculate trends
#     grouped = ds.groupby('time.month')

#     mon_list = []
#     wf_ft_list = []
#     fw_wt_list = []
#     gw_wt_list = []
#     nw_wt_list = []
#     for mon, dsmon in grouped:
#         dsfomon = ds_fstord.loc[dict(month=mon)]
#         wind_val = dsmon['PiC_UVnudge_LM2006'].values
#         force_val = dsmon['Anthro'].values
#         ghg_val = dsmon['GHG'].values
#         nonghg_val = dsmon['Non-GHG'].values
        
#         wind_time = dsfomon['PiC_UVnudge_LM2006'].values
#         force_time = dsfomon['Anthro'].values
        
#         # w/f * f/t
#         wind_mat = np.polyfit(force_val,wind_val,deg=1)
#         wind2nd = wind_mat[0]*force_time

#         # f/w * w/t
#         force2nd = (1./wind_mat[0])*wind_time

#         # g/w * w/t
#         ghg_mat = np.polyfit(wind_val,ghg_val,deg=1)
#         ghg2nd = ghg_mat[0]*wind_time

#         # n/w * w/t
#         nonghg_mat = np.polyfit(wind_val,nonghg_val,deg=1)
#         nonghg2nd = nonghg_mat[0]*wind_time

#         # Append
#         mon_list.append(mon)
#         wf_ft_list.append(wind2nd)
#         fw_wt_list.append(force2nd)
#         gw_wt_list.append(ghg2nd)
#         nw_wt_list.append(nonghg2nd)

#     month_2ndorder = xr.Dataset(
#         data_vars=dict(
#             wf_ft=(['month'],np.array(wf_ft_list)),
#             fw_wt=(['month'],np.array(fw_wt_list)),
#             gw_wt=(['month'],np.array(gw_wt_list)),
#             nw_wt=(['month'],np.array(nw_wt_list))
#         ),
#         coords=dict(
#         month=('month', np.array(mon_list))))
    
#     return month_2ndorder

#### CalcAnnTrd

In [27]:
def CalcAnnTrd(weighted_avg):
    # Calculate annual average
    ann_avg = CalcAnnSeasMean(weighted_avg, 1)

    # Calculate annual trend
    ann_trd = ann_avg.polyfit(dim='year',deg=1)
    ann_trd *= 10
    
    return ann_trd['polyfit_coefficients'].loc[dict(degree=1)]

#### CalcAnnTrd2ndOrder

In [28]:
# def CalcAnnTrd2ndOrder(ds, ds_fstord):
#     # Calculate annual average
#     ds_ann = CalcAnnMean(ds)

#     wind_val = ds_ann['PiC_UVnudge_LM2006'].values
#     force_val = (ds_ann['HIST_UVnudge_LM']-ds_ann['PiC_UVnudge_LM2006']).values
#     ghg_val = (ds_ann['GHG_UVnudge_LM']-ds_ann['PiC_UVnudge_LM2006']).values
#     nonghg_val = (ds_ann['HIST_UVnudge_LM']-ds_ann['GHG_UVnudge_LM']).values
#     ds_fstord['Anthro'] = ds_fstord['HIST_UVnudge_LM']-ds_fstord['PiC_UVnudge_LM2006']

#     wind_time = ds_fstord['PiC_UVnudge_LM2006'].values
#     force_time = ds_fstord['Anthro'].values
    
#     # w/f * f/t
#     wind_mat = np.polyfit(force_val,wind_val,deg=1)
#     wind2nd = wind_mat[0]*force_time

#     # f/w * w/t
#     force2nd = (1./wind_mat[0])*wind_time

#     # g/w * w/t
#     ghg_mat = np.polyfit(wind_val,ghg_val,deg=1)
#     ghg2nd = ghg_mat[0]*wind_time

#     # n/w * w/t
#     nonghg_mat = np.polyfit(wind_val,nonghg_val,deg=1)
#     nonghg2nd = nonghg_mat[0]*wind_time

#     ann_2ndorder = xr.Dataset(
#         data_vars=dict(
#             wf_ft=(['year'],np.array([wind2nd])),
#             fw_wt=(['year'],np.array([force2nd])),
#             gw_wt=(['year'],np.array([ghg2nd])),
#             nw_wt=(['year'],np.array([nonghg2nd]))
#         ),
#         coords=dict(
#         year=('year', np.array([t_domain]))))
    
#     return ann_2ndorder

#### CalcMonAnnTrend

In [29]:
def CalcMonAnnTrend(weighted_avg, time_type):
    # Calculate trends
    trd_list = []
    for dsname, da in weighted_avg.items():
        if time_type == 'm':
            da_trd = CalcMonthTrd(da)
        elif time_type == 'y':
            da_trd = CalcAnnTrd(da)
        trd_list.append(da_trd.rename(dsname))

    ds_trd = xr.merge(trd_list)
            
    return ds_trd

#### CalcMonAnn2ndOrder

In [30]:
# def CalcMonAnn2ndOrder(weighted_avg, firstorder, time_type):
#     # Calculate 2nd order terms
#     if time_type == 'm':
#         ds_2ndtrd = CalcMonthTrd2ndOrder(weighted_avg, firstorder)
#     elif time_type == 'y':
#         ds_2ndtrd = CalcAnnTrd2ndOrder(weighted_avg, firstorder)
            
#     return ds_2ndtrd

#### CalcXXyrMonAnnTrend

In [31]:
def CalcXXyrMonAnnTrend(ds, time_type):
    
    startyrs = np.arange(t_domain,t_end-(trd_length-2))
    endyrs = np.arange(t_domain+(trd_length-1),t_end+1)
    trd_list = []
    for sy in startyrs:
        print('Calculating trends for '+str(sy)+'-'+str(sy+trd_length-1))
        slice_time_i = dict(time=slice(str(sy)+'-01-01',str(sy+trd_length-1)+'-12-31'))
        ds_sub = copy.deepcopy(ds.loc[slice_time_i])

        dss_trd = CalcMonAnnTrend(ds_sub, time_type)
        trd_list.append(dss_trd)
        
    ds_trd = xr.concat(trd_list, dim=pd.Index(startyrs, name='start_year'))
    ds_trd = ds_trd.assign_coords(end_year=('start_year', endyrs))

    return ds_trd

#### AttrCalc

In [32]:
def AttrCalc(ds, varname, skip_obs):
    use_obs = varname != 'MOC' and not skip_obs
    if use_obs:
        obs_name_list = [name for name, attrs in ds_names.items() if (attrs[0] and attrs[1] == 3)]

    funcs = {0: np.subtract,
            1: np.add}

    # 0: subtract, 1: addition
    attr_names = {'Non-GHG': ['HIST_UVnudge_LM',0,'GHG_UVnudge_LM'],
                 'Anthropogenic forcing': ['HIST_UVnudge_LM',0,'PiC_UVnudge_LM2006'],
                 'Greenhouse gases': ['GHG_UVnudge_LM',0,'PiC_UVnudge_LM2006'],
                 'Winds': ['PiC_UVnudge_LM2006'],
                 'HIST+winds': ['HIST_UVnudge_LM']}
    if use_obs:
        for obs_name in obs_name_list:
            attr_names[ds_paper_names[obs_name]] = [obs_name]
            attr_names['Bias ('+ds_paper_names[obs_name][5:-1]+')'] = [obs_name,0,'HIST_UVnudge_LM']
    
    attr_list = []
    
    for aname, eq in attr_names.items():
        print(aname)
        # If term composed of one variable - no calculation needed
        if len(eq) == 1 and ds_names[eq[0]][0]:
            eqname = eq[0]+' mean' if ds_names[eq[0]][1] == 0 else eq[0]
            attr_list.append(ds[eqname].rename(aname))
            print('  Calculated')

        # If calculation needs two terms
        elif len(eq) > 1:
            # Find all variable names in calculation
            calc_names = eq[::2]

            # See if all names in calculation are set to be plotted, if not don't calculate this attribution term
            all = True
            for cname in calc_names:
                if not ds_names[cname][0]:
                    all = False
                    break
            if not all:
                continue
            else:
                eqname = eq[0] + ' mean' if ds_names[eq[0]][1] == 0 else eq[0]
                da_attr = ds[eqname]
                for i in np.arange(1,len(eq)-1,2):
                    eqname = eq[i+1] + ' mean' if ds_names[eq[i+1]][1] == 0 else eq[i+1]
                    da_attr = xr.apply_ufunc(funcs[eq[i]],da_attr,ds[eqname], dask='allowed')
                attr_list.append(da_attr.rename(aname))
                print('  Calculated')

    ds_attr = xr.merge(attr_list)
    return ds_attr

#### CreateRegridder

In [33]:
def CreateRegridder(rtype):
    # rtype: 'cice' or 'era'
    filestem = {'cice': 'CICEregridder',
                'era': 'ERAregridder'}

    method = {'cice': 'nearest_s2d',
              'era': 'bilinear'}

    rlats = {False: arclats, # Arctic
             True: lats} # Global
    slicer = {'cice': {'ice': slice_icemod},
              'era': {'ice': slice_iceobs,
                     'atm': slice_atmspt}}
    
    # Check for existing target grid
    targetgridpath = {'cice': path_to_plotdata+filestem['cice']+'.'+sd_str+'.targetgrid.nc',
                      'era': path_to_plotdata+filestem['era']+'.'+sd_str+'.targetgrid.nc'}

    # Check for existing sample grid
    samplegridpath = {'cice': path_to_plotdata+filestem['cice']+'.'+sd_str+'.samplegrid.nc',
                      'era': path_to_plotdata+filestem['era']+'.'+sd_str+'.samplegrid.nc'}

     # File with sample grid
    samplegridpath_in = {'cice': path_to_lensdata+'b.e21.B1850.f09_g17.CMIP6-piControl.001.h.hi.000801-008212.nc',
                         'era': path_to_work+'processed_ERA5_data/ERA5.h0.TREFHT.195001-202412.nc'}

    # Check for existing regridder weights
    weightpath = {'cice': path_to_plotdata+filestem['cice']+'.'+sd_str+'.weights.nc',
                'era': path_to_plotdata+filestem['era']+'.'+sd_str+'.weights.nc'}

    # If exists, open file for target grid
    if os.path.isfile(targetgridpath[rtype]):
        target_grid = xr.open_dataset(targetgridpath[rtype])
        
    # Else, create and save file
    else:
        # Create target cice grid
        if rtype == 'cice':
            lon2d, lat2d = np.meshgrid(lons, rlats[s_domain])
            target_grid = xr.Dataset({'lat': (['y', 'x'], lat2d),'lon': (['y', 'x'], lon2d)})
    
        # Create target era grid
        elif rtype == 'era':
            target_grid = xr.Dataset({'lat': ('y', rlats[s_domain]), 'lon': ('x', lons)})
            
        target_grid.to_netcdf(targetgridpath[rtype])

    # If exists, open file
    if os.path.isfile(samplegridpath[rtype]):
        sample_grid = xr.open_dataset(samplegridpath[rtype])
    
    # Else, create and save file
    else:
        sample_grid = xr.open_dataset(samplegridpath_in[rtype])
        sample_grid = sample_grid[dict(time=0)]
        sample_grid = sample_grid if s_domain else sample_grid.loc[slicer[rtype][comp]]
        sample_grid.to_netcdf(samplegridpath[rtype])
   

    # If weights exists, create regridder
    if os.path.isfile(weightpath[rtype]):
        regridder = xe.Regridder(sample_grid, target_grid, method[rtype], filename=weightpath[rtype],reuse_weights=True)

    # Else, create regridder and save weights
    else:
        regridder = xe.Regridder(sample_grid, target_grid, method[rtype], reuse_weights=False)
        regridder.to_netcdf(weightpath[rtype])
    
    return regridder

#### Regrid

In [34]:
def Regrid(ds_timeavg, regridder, regrid_type):
    # regrid_type: 'cice' or 'era'
    cice_cond = (regrid_type == 'cice')
    era_cond = (regrid_type == 'era')

    if cice_cond:
        print('Regridding CICE grid -> ATM grid...')
    if era_cond:
        print('Regridding ERA5 grid -> ATM grid...')

    # Do fillna if regridding sic
    if cice_cond:
        nval = 0.000001

    # If regridding ERA5 DataArray
    if era_cond and type(ds_timeavg) == xr.core.dataarray.DataArray:
        da = ds_timeavg.rename({'latE': 'lat', 'lonE': 'lon'})
        da_re = regridder(da)
        da_re = da_re.rename({'x':'lon', 'y':'lat'})
        da_regrid = da_re.rename('ERA5')

        return da_regrid
        

    # Else, regridding dataset
    else:
        # Regrid data
        regrid_list = []
        for dsname, da in ds_timeavg.items():
            cond = {'cice': dsname == 'ERA5',
                    'era': dsname != 'ERA5'}
            # If CICE regridding & dataset name is ERA5, don't regrid ERA5
            # If ERA regridding & dataset name is not ERA5, don't regrid, just append
            if cond[regrid_type]:
                # Don't regrid, just add for era regridding
                if era_cond:
                    da = da.assign_coords({'lon': lons, 'lat': arclats})
                    regrid_list.append(da.rename(dsname))
            
            else:
                print('Regridding '+dsname)
    
                # Need to rename for era
                if era_cond:
                    da = da.rename({'latE': 'lat', 'lonE': 'lon'})
                da_re = regridder(da)
        
                # Rename x, y to be lon, lat and reassign lon and lat data (which DOES NOT include cyclic point)
                da_re = da_re.rename({'x':'lon', 'y':'lat'})
                da_re = da_re.assign_coords({'lon':lons, 'lat': arclats})
    
                # Only fillna for sic
                if cice_cond:
                    da_re = da_re.fillna(nval)
                    
                regrid_list.append(da_re.rename(dsname))
    
        # Fill na for ERA5 only if sic 
        if cice_cond and ds_names['ERA5'][0]:
            regrid_list.append(ds_timeavg['ERA5'].fillna(nval))
        
        ds_regrid = xr.merge(regrid_list, join='left')
    
        return ds_regrid

#### CalcGridArea

In [35]:
def CalcGridArea(ds):
    # For ERA5 0.25x0.25 grid
    dlat = 0.25
    dlon = 0.25
    R = 6367.47 # km

    lons, lats = np.meshgrid(ds.lon.values, ds.lat.values)

    dy = R*np.deg2rad(dlat)
    dx = np.deg2rad(dlon)*R*np.cos(np.deg2rad(lats))

    area = np.abs(dx*dy)
    ds = ds.assign_coords(area=(('lat','lon'), area))

    return ds

#### CalcSIE

In [36]:
def CalcSIE(ds, set_type):
    # Extracts aice variable and only counts cells with sic > 15%
    ds_aice = ds
    ds_aice = xr.where(ds_aice > .15,1,0)

    # Multiples selected sic cells with tarea, then only selects Arctic sea ice, sums over the entire domain, then converts to km2
    if set_type == 3:
        dsa = (ds_aice*ds['area']).sum(dim=['lon','lat'])*1e-6
    else:
        dsa = (ds_aice*ds['tarea']).sum(dim=['ni','nj'])*1e-6*1e-6

    # Modifies attributes of DataArray accordingly
    dsa.attrs['units'] = 'million km^2'
    dsa.attrs['long_name'] = 'Arctic sea ice area'

    # Assigns new variable to dataset and returns original dataset
    return dsa

#### CalcTrend

In [37]:
def CalcTrend(da, time_type):
    da_b = da.dropna(dim='time')
    slope, intercept, rval, pval, stderr = xr.apply_ufunc(stats.linregress,
                                                          da_b[time_type], da_b,
                                                          input_core_dims=[[time_type], [time_type]],
                                                          output_core_dims=[[],[],[],[],[]],
                                                          output_dtypes=[xr.core.dataarray.DataArray,xr.core.dataarray.DataArray,xr.core.dataarray.DataArray,
                                                                        xr.core.dataarray.DataArray,xr.core.dataarray.DataArray],
                                                          vectorize=True,
                                                          dask='parallelized',
                                                          dask_gufunc_kwargs=dict(allow_rechunk=True))
    slope = slope*10
    return slope, pval

#### AddCoordTrend

In [38]:
def AddCoordTrend(da, time_type):
    da_sizes = da.sizes
    new_time = np.arange(1,da_sizes[time_type]+1)
    if time_type == 'time':
        da = da.assign_coords(time=new_time)
    elif time_type == 'year':
        da = da.assign_coords(year=new_time)
    return da

#### AddCyclic

In [39]:
def AddCyclic(da: xr.DataArray, londim) -> xr.DataArray:
    # Add cyclic point
    cyclic_data, cyclic_lon = add_cyclic_point(da.data, coord=da[londim])
    cyclic_coords = {dim: da.coords[dim] for dim in da.dims}
    cyclic_coords[londim] = cyclic_lon

    da = xr.DataArray(cyclic_data, dims=da.dims, coords=cyclic_coords, attrs=da.attrs, name=da.name)
    return da

#### VertSptrendsLoop

In [40]:
def VertSptrendsLoop(ds_sptpl, time_type, varname):
    ds_sptpl = ds_sptpl.groupby('plev')
    
    lev_spt_list = []
    for lev, dsp_lev in ds_sptpl:
        print('Calculating spatial trends for '+str(lev)+' hPa')
        dlev_trends = SpatZonTrends(dsp_lev, time_type, varname)
        lev_spt_list.append(dlev_trends)
        
    ds_pltrends = xr.concat(lev_spt_list, dim='plev')

    return ds_pltrends

#### SpatZonTrends

In [41]:
def SpatZonTrends(ds, time_type, varname):
    # Bool for spatial/zonal differentiation
    ds_spz = ds
    dimlist = list(ds_spz.coords)
    zonspt_bool = 'lon' in dimlist # True: spatial, False: zonal
    
    # Varname bool
    var_bool = varname == 'Z3'
    
    # Calculating averages
    if time_avg == 0:
        ds_spz = ds_spz.groupby('time.month')
        
    # Seasonal averaging
    elif time_avg == 2:
        ds_spz = ds_spz.resample(time='QS-DEC').mean('time')
        ds_spz = ds_spz.groupby('time.month')
    
    # Calculate trends
    trends_tm_list = []
    
    # Cycle through months/seasons
    for m, ds in ds_spz:
        
        print('Trends for '+str(m))
        trend_ds_list = []

        # Add linear time coordinate for trends - doesn't work with datetime time coordinate
        ds = AddCoordTrend(ds, time_type)

        # Cycle through all variables
        for dsname, da in ds.items():
            print('  Calculating trends for '+dsname)
            da_gp_trend = CalcTrend(da.load(), time_type) 
            
            trend_ds_list.append(da_gp_trend.rename(dsname))

            era_bool = dsname == 'ERA5'
            
            # If 3D variable & ERA5, regrid data to CESM2
            if era_bool and zonspt_bool:
                # If Z3 (i.e. not on CESM2 grid)
                if var_bool:
                    da_gp_trend = Regrid(da_gp_trend, regridderERA, 'era')
                else:
                    da_gp_trend = da_gp_trend.rename({'latE': 'lat', 'lonE': 'lon'})
            # If ERA5 & zonal (and thus not including Z3)
            elif era_bool and not zonspt_bool:
                da_gp_trend = da_gp_trend.rename({'latE': 'lat'})
            
        ds_trend_slice = xr.merge(trend_ds_list, compat='no_conflicts')
        trends_tm_list.append(ds_trend_slice)

    ds_trend = xr.concat(trends_tm_list, dim=pd.Index(date_str, name=time_outstr))

    return ds_trend

#### PattCorr

In [42]:
def PattCorr(ds_trend):
    ## Calculation pattern correlation coefficient
    ds_trend = ds_trend.groupby(time_outstr)

    # Calculate pattern correlation
    pcorr_m_list = []

    for m, ds in ds_trend:
        print('Pattern correlation for '+str(m))
        pcorr_ds_list = []

        # Pull out ERA5
        da_era = ds['ERA5']

        weights = np.cos(np.deg2rad(ds.lat))

        for dsname, da in ds.items():
            if dsname != 'ERA5':
                # Calculate corr
                da_pcorr = xr.corr(da_era, da, weights=weights)

                pcorr_ds_list.append(da_pcorr.rename(dsname))

        ds_pcorr_slice = xr.merge(pcorr_ds_list)
        pcorr_m_list.append(ds_pcorr_slice)


    ds_pcorr = xr.concat(pcorr_m_list, dim=pd.Index(date_str, name=time_outstr))
    return ds_pcorr

#### SpatZonAvg

In [43]:
def SpatZonAvg(ds):
    # Calculating averages
    if time_avg == 0:
        ds_avg = ds.groupby('time.month').mean('time')
        ds_avg = ds_avg.assign_coords(month=mon_str)
        
    # Seasonal averaging
    elif time_avg == 2:
        month_length = ds.time.dt.days_in_month

        # Calculate the weights by grouping by 'time.season'.
        weights = (month_length.groupby("time.season")/month_length.groupby("time.season").sum())

        # Calculate the weighted average
        ds_avg = (ds * weights).groupby("time.season").sum(dim="time")

    return ds_avg

#### SpatialVar

In [44]:
def SpatialVar(ds):
    # Calculating variances
    if time_avg == 0:
        ds_var = ds.groupby('time.month').var('time')
        
    # Seasonal variances
    elif time_avg == 2:
        ds_avg = ds.resample(time='QS-DEC').mean('time')
        ds_var = ds_avg.groupby('time.month').var('time')
        ds_var = ds_var.assign_coords(month=seas_str)
        ds_var = ds_var.rename({'month':'season'})

    return ds_var

#### AddAllCyclic

In [45]:
def AddAllCyclic(ds_trend):
    londim = defaultdict(lambda: 'lon')
    latdim = defaultdict(lambda: 'lat')

    londim['ERA5'] = 'lonE'
    latdim['ERA5'] = 'latE'
        
    cyclic_ds_list = []
        
    for dsname, da in ds_trend.items():
        LEcheck = dsname in ds_names and ds_names[dsname][1] == 0 # True: large ensemble with slice dimension
        if file_bool:
            if LEcheck:
                da = da.transpose(time_outstr,'slice',latdim[dsname],londim[dsname])
            else:
                da = da.transpose(time_outstr, latdim[dsname],londim[dsname])
        else:
            if LEcheck:
                da = da.transpose(time_outstr,'slice','plev',latdim[dsname],londim[dsname])
            else:
                da = da.transpose(time_outstr,'plev',latdim[dsname],londim[dsname])
        da_cyc = AddCyclic(da, londim[dsname])
        
        cyclic_ds_list.append(da_cyc.rename(dsname))
            
    ds_trend = xr.merge(cyclic_ds_list)

    return ds_trend

#### SaveData

In [46]:
def SaveData(ds, plot_type, varname, tavg, plot_level=None):
    level_str = '' if plot_level == None else 'Z'+str(int(plot_level))+'.'
    domain_str = sd_str
    if r_domain:
        domain_str += '.'+rd_str
    if o_domain:
        domain_str += '.'+od_str
    
    filename = plot_type+'.'+varname+'.'+domain_str+'.'+level_str+td_str+'.'+tavg+'.'+ens_str+'.nc'
    
    print('Saving '+filename)
    # File format is:
    # (plot type, including special averaging, i.e. anomalies).varname.spatialdomain.timedomain.timeaveraging.ensemble_type.nc
    ds.to_netcdf(path_to_plotdata+filename, format='NETCDF4')
    
    return None

### Process and load data

In [47]:
%%time

ds_proc = CreateMasterDS(var)

Figure out processing calculations to do
LENS2 piControl
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
LENS2 HIST-SSP370
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
GHG2
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
PiC_UVnudge_LM2006
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
HIST_UVnudge_LM
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
GHG_UVnudge_LM
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Calculating maximum
  Processing on dataset complete
RAPID
  Initial data loading complete
  Fixed time dimension
  Sliced data
  Processing on dataset complete
All datasets merge

In [48]:
%%time

# Add U & V variables if doing spatial trends or patterns and original variable is geopotential
if (plot_types['zonal'][0] or plot_types['spatial'][0]) and var == 'Z3':
    # Create U & V datasets
    ds_u = CreateMasterDS('U')
    ds_v = CreateMasterDS('V')

CPU times: user 4 µs, sys: 1e+03 ns, total: 5 µs
Wall time: 6.91 µs


### Plotting set-up

In [49]:
%%time

mon_str = np.array(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
seas_str = np.array(['MAM','JJA','SON','DJF'])


CPU times: user 27 µs, sys: 4 µs, total: 31 µs
Wall time: 32.2 µs


## Month trend plots
### Set up

In [50]:
if plot_types['mtrd'][0]:
    graph_type_str = 'Linear.Trend'
    ## Select plot type - timeseries or monthly - to make and assign variables accordings
    

    # Timeseries
    if time_avg == 4:
        dim_avg = 'time.month'
        period = 'month'
        trdyr_str = '.'+str(trd_length)+'yr'

### Data processing

In [51]:
%%time

if plot_types['mtrd'][0]:
    ## GO BACK TO POLYFIT & REMOVE P-vals
    
    if subset_time:
        # Calculate monthly trends
        ds_mon_trd = CalcMonAnnTrend(ds_proc, 'm')
    
        # Calculate annual trends
        ds_ann_trd = CalcMonAnnTrend(ds_proc, 'y')
    
        # Write out data
        SaveData(ds_mon_trd, graph_type_str, var, 'month')
        SaveData(ds_ann_trd, graph_type_str, var, 'year')

    else:
        if var != 'MOC':
            ds_mon_trd = CalcXXyrMonAnnTrend(ds_proc, 'm')
                
            SaveData(ds_mon_trd, graph_type_str+trdyr_str, var, 'month')
            
        ds_ann_trd = CalcXXyrMonAnnTrend(ds_proc, 'y')
            
        SaveData(ds_ann_trd, graph_type_str+trdyr_str, var, 'year')

Calculating trends for 1950-1969
Calculating trends for 1951-1970
Calculating trends for 1952-1971
Calculating trends for 1953-1972
Calculating trends for 1954-1973
Calculating trends for 1955-1974
Calculating trends for 1956-1975
Calculating trends for 1957-1976
Calculating trends for 1958-1977
Calculating trends for 1959-1978
Calculating trends for 1960-1979
Calculating trends for 1961-1980
Calculating trends for 1962-1981
Calculating trends for 1963-1982
Calculating trends for 1964-1983
Calculating trends for 1965-1984
Calculating trends for 1966-1985
Calculating trends for 1967-1986
Calculating trends for 1968-1987
Calculating trends for 1969-1988
Calculating trends for 1970-1989
Calculating trends for 1971-1990
Calculating trends for 1972-1991
Calculating trends for 1973-1992
Calculating trends for 1974-1993
Calculating trends for 1975-1994
Calculating trends for 1976-1995
Calculating trends for 1977-1996
Calculating trends for 1978-1997
Calculating trends for 1979-1998
Calculatin

## Line plots
### Set up

In [52]:
if plot_types['line'][0]:
    graph_type_str = 'Linear'
        

    # Time averaging for yearly plot
    if time_avg == 1:
        dim_avg = 'time.year'  
        period='year'
    if time_avg == 2:
        dim_avg = 'time.season'
        period = 'season'
    if time_avg == 4:
        dim_avg = 'time.month'
        period ='time'

### Data processing

In [53]:
%%time

if plot_types['line'][0]:

    ## Absolute (only one that will be used for SIA-one month, TOA, AMOC)
    # Yearly averaging for absolute
    if time_avg == 1:
        ds_abs = CalcAnnSeasMean(ds_proc, 1)
        CalcLEMinMax(ds_abs)
        
        SaveData(ds_abs, graph_type_str+'.abs', var, period)

    elif time_avg == 2:
        ds_abs = CalcAnnSeasMean(ds_proc, 2)
        CalcLEMinMax(ds_abs)
        
        SaveData(ds_abs, graph_type_str+'.abs', var, period)

    elif time_avg == 4:
        # If only plotting one month from SIA data
        ds_abs = ds_proc.groupby(dim_avg)

        for m, ds in ds_abs:
            CalcLEMinMax(ds)

            SaveData(ds, graph_type_str+'.abs', var, mon_str[m-1])

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 7.39 µs


In [54]:
%%time

if plot_types['line'][0]:
    # If sea ice area, make absolute correlation coefficients
    if time_avg == 4:
        # Anomalies
        anom_list = []
        for m, ds in ds_abs:
            da_anom = CalcAnom(ds, period)
            CalcLEMinMax(da_anom)
            
            anom_list.append(da_anom)
            SaveData(da_anom, graph_type_str+'.anom', var, mon_str[m-1])
            
        ds_anom = xr.concat(anom_list, dim='time')
        ds_anom = ds_anom.groupby(dim_avg)

        # Detrended anomalies
        dtrd_anom_list = []
        for m, ds in ds_anom:
            dtrd_m_list = []
            for dsname, da in ds.items():
                da_dtrd = CalcDetAnom(da, period)
                dtrd_m_list.append(da_dtrd)
                
            da_dtrd = xr.merge(dtrd_m_list)
            CalcLEMinMax(da_dtrd)
            
            dtrd_anom_list.append(da_dtrd)
            SaveData(da_dtrd, graph_type_str+'.anomdtrd', var, mon_str[m-1])
            
        ds_dtrd = xr.concat(dtrd_anom_list, dim='time')
        ds_dtrd = ds_dtrd.groupby(dim_avg)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 7.39 µs


In [55]:
%%time

if plot_types['line'][0] and time_avg == 4 and use_era5:
    # Calculate R for each month
    for m, ds in ds_abs:
        abs_r_list = []
        era5_abs = ds['ERA5']
        for dsname, attrs in ds_names.items():
            if attrs[0] and (attrs[1] == 1 or attrs[1] == 2):
                da_abs_r = CalcR(era5_abs, ds[dsname], period, attrs[1])
                abs_r_list.append(da_abs_r.rename(dsname))
                
        ds_abs_r = xr.merge(abs_r_list)
        SaveData(ds_abs_r, graph_type_str+'.absR', var, mon_str[m-1])
    
    # Calculate R for anomalies
    for m, ds in ds_anom:
        anom_r_list = []
        era5_anom = ds['ERA5']
        for dsname, attrs in ds_names.items():
            if attrs[0] and (attrs[1] == 1 or attrs[1] == 2):
                da_anom_r = CalcR(era5_anom, ds[dsname], period, attrs[1])
                anom_r_list.append(da_anom_r.rename(dsname))
                
        ds_anom_r = xr.merge(anom_r_list)
        SaveData(ds_anom_r, graph_type_str+'.anomR', var, mon_str[m-1])

    # Calculate R for detrended anomalies
    for m, ds in ds_dtrd:
        dtrd_r_list = []
        era5_dtrd = ds['ERA5']
        for dsname, attrs in ds_names.items():
            if attrs[0] and (attrs[1] == 1 or attrs[1] == 2):
                da_dtrd_r = CalcR(era5_dtrd, ds[dsname], period, attrs[1])
                dtrd_r_list.append(da_dtrd_r.rename(dsname))
                
        ds_dtrd_r = xr.merge(dtrd_r_list)
        SaveData(ds_dtrd_r, graph_type_str+'.anomdtrdR', var, mon_str[m-1])

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.91 µs


In [56]:
%%time

if plot_types['line'][0] and var == 'TREFHT':
    # If surface temperature, make anomaly, detrended anomaly (yearly only), and ensemble spread plots
    # Calculate R for absolute temperature
    abs_r_list = []
    era5_abs = ds_abs['ERA5']
    for dsname, attrs in ds_names.items():
        if attrs[0] and (attrs[1] == 1 or attrs[1] == 2 or attrs[1] == 4):
            da_abs_r = CalcR(era5_abs, ds_abs[dsname], period, attrs[1])
            abs_r_list.append(da_abs_r.rename(dsname))
            
    ds_abs_r = xr.merge(abs_r_list)
    SaveData(ds_abs_r, graph_type_str+'.absR', var, period)
    
    # Anomalies
    ds_anom = CalcAnom(ds_abs, period)
    CalcLEMinMax(ds_anom)
    
    SaveData(ds_anom, graph_type_str+'.anom', var, period)

    # Calculate R for anomalies
    anom_r_list = []
    era5_anom = ds_anom['ERA5']
    for dsname, attrs in ds_names.items():
        if attrs[0] and (attrs[1] == 1 or attrs[1] == 2 or attrs[1] == 4):
            da_anom_r = CalcR(era5_anom, ds_anom[dsname], period, attrs[1])
            anom_r_list.append(da_anom_r.rename(dsname))
            
    ds_anom_r = xr.merge(anom_r_list)
    SaveData(ds_anom_r, graph_type_str+'.anomR', var, period)

    # Ensemble spread
    if not ens_type:
        enspd_list = []
        for dsname, attrs in ds_names.items():
            if attrs[0] and attrs[1] == 1:
                da_enspd = CalcEnsSp(ds_abs[dsname])
                enspd_list.append(da_enspd.rename(dsname))
                
        ds_enspd = xr.merge(enspd_list)
        SaveData(ds_enspd, graph_type_str+'.espd', var, period)

    # Detrended anomalies only if yearly
    # Detrended anomalies
    dtrd_anom_list = []
    for dsname, da in ds_anom.items():
        da_dtrd = CalcDetAnom(da, period)
        dtrd_anom_list.append(da_dtrd.rename(dsname))

    ds_dtrd = xr.merge(dtrd_anom_list)
    CalcLEMinMax(ds_dtrd)

    SaveData(ds_dtrd, graph_type_str+'.anomdtrd', var, period)

    # Calculate R for detrended anomalies
    dtrd_r_list = []
    era5_dtrd = ds_dtrd['ERA5']
    for dsname, attrs in ds_names.items():
        if attrs[0] and (attrs[1] == 1 or attrs[1] == 2 or attrs[1] == 4):
            da_dtrd_r = CalcR(era5_dtrd, ds_dtrd[dsname], period, attrs[1])
            dtrd_r_list.append(da_dtrd_r.rename(dsname))

    ds_dtrd_r = xr.merge(dtrd_r_list)
    SaveData(ds_dtrd_r, graph_type_str+'.anomdtrdR', var, period)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 7.15 µs


## Create Regridders

In [57]:
%%time
if plot_types['spatial'][0]:
    ## Only run after time averaging!!!
    
    # NEITHER REGRIDDER INCLUDES CYCLIC POINT!
    
    if comp == 'ice':
        ## Create Regridder for CICE grid->CESM2
        regridderCICE = CreateRegridder('cice')
    
    if plots['strd'][0]:
        ## Create Regridder for ERA5->CESM2
        regridderERA = CreateRegridder('era')

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 7.15 µs


In [58]:
# Spatial plots
if plot_types['spatial'][0]:
    # Monthly
    if time_avg == 0:
        period = 'time'
        date_str = mon_str

    # Seasonally
    elif time_avg == 2:
        period = 'time'
        date_str = seas_str

    # Yearly
    elif time_avg == 1:
        period = ''

## Spatial trend plots

### Set up

In [59]:
if plots['strd'][0]:
    graph_type_str = 'Map.Trend'

### Data processing

In [60]:
%%time

if plots['strd'][0]:
    # If not 3D
    if file_bool:
        
        # Calculate trends
        print('Calculating spatial trends in '+var)
        ds_sptrend = SpatZonTrends(ds_proc,  period, var)

    else:
        ds_proc = ds_proc.where(ds_proc.plev.isin(plot_levels), drop=True)  
        ds_u = ds_u.where(ds_u.plev.isin(plot_levels), drop=True)
        ds_v = ds_v.where(ds_v.plev.isin(plot_levels), drop=True)
        
        # Calculate trends
        print('Calculating spatial trends in '+var)
        ds_sptrend = VertSptrendsLoop(ds_proc, period, var)
        print('Calculating spatial trends in U')
        ds_usptrend = VertSptrendsLoop(ds_u, period, 'U')
        print('Calculating spatial trends in V')
        ds_vsptrend = VertSptrendsLoop(ds_v, period, 'V')

CPU times: user 6 µs, sys: 0 ns, total: 6 µs
Wall time: 7.87 µs


In [61]:
%%time

if plots['strd'][0]:
    if comp == 'ice':
        ds_sptrend = Regrid(ds_sptrend, regridderCICE, 'cice')
        
    if use_era5:  
        # Regrid ERA5 data
        ds_sptrend_corr = Regrid(ds_sptrend, regridderERA, 'era')
    
        # Calculate pattern coefficients between ERA5-PiC_UVnudgeX
        ds_trdcor = PattCorr(ds_sptrend_corr)
    
        SaveData(ds_trdcor, graph_type_str+'.pcorr', var, time_outstr)

    if var == 'Z3':
        # Rename ERA5 lat/lon
        ds_era = ds_usptrend['ERA5']
        ds_era = ds_era.rename({'latE':'lat', 'lonE':'lon'})
        ds_usptrend_corr = ds_usptrend.drop_vars('ERA5')
        ds_usptrend_corr = ds_usptrend_corr.drop_dims(['latE','lonE'])
        ds_usptrend_corr = ds_usptrend_corr.assign({'ERA5':ds_era})

        ds_era = ds_vsptrend['ERA5']
        ds_era = ds_era.rename({'latE':'lat', 'lonE':'lon'})
        ds_vsptrend_corr = ds_vsptrend.drop_vars('ERA5')
        ds_vsptrend_corr = ds_vsptrend_corr.drop_dims(['latE','lonE'])
        ds_vsptrend_corr = ds_vsptrend_corr.assign({'ERA5':ds_era})
        
        # Calculate pattern coefficients between ERA5-PiC_UVnudgeX
        ds_utrdcor = PattCorr(ds_usptrend_corr)
        ds_vtrdcor = PattCorr(ds_vsptrend_corr)
    
        SaveData(ds_utrdcor, graph_type_str+'.pcorr', 'U', time_outstr)
        SaveData(ds_vtrdcor, graph_type_str+'.pcorr', 'V', time_outstr)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 7.15 µs


In [62]:
%%time

if plots['strd'][0]:
    # Add cyclic data
    ds_sptrend = AddAllCyclic(ds_sptrend)
    
    SaveData(ds_sptrend, graph_type_str, var, time_outstr)

    if var == 'Z3':
        ds_usptrend = AddAllCyclic(ds_usptrend)

        ds_vsptrend = AddAllCyclic(ds_vsptrend)

        SaveData(ds_usptrend, graph_type_str, 'U', time_outstr)

        SaveData(ds_vsptrend, graph_type_str, 'V', time_outstr)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.91 µs


## Spatial plots
### Set up

In [63]:
if plots['map'][0]:
    graph_type_str = 'Map'

### Data processing

In [64]:
%%time

if plots['map'][0]:
    # Calculating averages
    ds_sp = SpatZonAvg(ds_proc)

    if var == 'Z3':
        ds_spv = SpatZonVar(ds_proc)
        ds_usp = SpatZonAvg(ds_u)
        ds_vsp = SpatZonAvg(ds_v)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.68 µs


In [65]:
%%time

if plots['map'][0]:
    if comp == 'ice':
        ds_sp = Regrid(ds_sp, regridderCICE, 'cice')
        
    # Add cyclic data 
    ds_sp = AddAllCyclic(ds_sp)
    SaveData(ds_sp, graph_type_str, var, time_outstr)
    
    if var == 'Z3':
        ds_spv = AddAllCyclic(ds_spv)
        ds_usp = AddAllCyclic(ds_usp)
        ds_vsp = AddAllCyclic(ds_vsp)

        SaveData(ds_spv, graph_type_str+'.var', var, time_outstr)
        SaveData(ds_usp, graph_type_str, 'U', time_outstr)
        SaveData(ds_vsp, graph_type_str, 'V', time_outstr)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 7.39 µs


## Zonal trend plots
### Set up

In [66]:
# Zonal plots
if plot_types['zonal'][0]:
    # Monthly
    if time_avg == 0:
        period = 'time'
        date_str = mon_str

    # Seasonally
    elif time_avg == 2:
        period = 'time'
        date_str = seas_str

In [67]:
if plots['ztrd'][0]:
    graph_type_str = 'Zonal.Trend'

### Data processing

In [68]:
%%time

if plots['ztrd'][0]:
    # Calculate statistics for LENS2 piControl
    ds_lens_stats = LENS2TrendsEnsemble(graph_type_str, ds_proc['LENS2 piControl'], period, var, False)
    ds_ulens_stats = LENS2TrendsEnsemble(graph_type_str, ds_u['LENS2 piControl'], period, 'U', False)
    ds_vlens_stats = LENS2TrendsEnsemble(graph_type_str, ds_v['LENS2 piControl'], period, 'V', False)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.44 µs


In [69]:
%%time

if plots['ztrd'][0]:
    print('Calculating zonal trends in '+var)
    ds_ztrend, ds_zpval = SpatZonTrends(ds_proc, ds_lens_stats, period, var)

    if var == 'Z3':
        print('Calculating zonal trends in U')
        ds_uztrend, ds_uzpval = SpatZonTrends(ds_u, ds_ulens_stats, period, 'U')
        print('Calculating zonal trends in V')
        ds_vztrend, ds_vzpval = SpatZonTrends(ds_v, ds_vlens_stats, period, 'V')    

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 6.91 µs


In [70]:
%%time

if plots['ztrd'][0]:
    SaveData(ds_ztrend, graph_type_str, var, time_outstr)
    SaveData(ds_zpval, graph_type_str+'.pval', var, time_outstr)

    if var == 'Z3':
        SaveData(ds_uztrend, graph_type_str, 'U', time_outstr)
        SaveData(ds_uzpval, graph_type_str+'.pval', 'U', time_outstr)
        
        SaveData(ds_vztrend, graph_type_str, 'V', time_outstr)
        SaveData(ds_vzpval, graph_type_str+'.pval', 'V', time_outstr)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.68 µs


## Zonal Plots
### Set up

In [71]:
if plots['zon'][0]:
    graph_type_str = 'Zonal'

### Data processing

In [72]:
%%time

if plots['zon'][0]:
    # Time averaging
    ds_zon = SpatZonAvg(ds_proc)
    
    if var == 'Z3':
        # Calculate for U & V
        ds_uzon = SpatZonAvg(ds_u)
        ds_vzon = SpatZonAvg(ds_v)    

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 6.91 µs


In [73]:
%%time

if plots['zon'][0]:
    SaveData(ds_zon, graph_type_str, var, time_outstr)

    if var == 'Z3':
        SaveData(ds_uzon, graph_type_str, 'U', time_outstr)
        SaveData(ds_vzon, graph_type_str, 'V', time_outstr)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.44 µs


In [ ]:
client.shutdown()